# 9 · Materials in 3D — a cup of coffee ☕

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=09-materials-3d.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/09-materials-3d.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::


Notebook 8 named regions on a flat 2D cookie. The same ideas shine in **3D**,
where a body is built from **several solids** glued together — each with its own
material — and the boundaries split into faces with different roles. We build a
real **mug filled with coffee**: a ceramic shell around a body of liquid, and
solve a steady heat problem with **piecewise material coefficients** and
**mixed boundary conditions**.

**How the mug is built (OpenCASCADE).** The mug is **axisymmetric**, so it is a
body of **revolution** — a 2D cross-section spun $360^\circ$ about the vertical
axis. In OCC a `Cylinder` *is* exactly that (a rectangle revolved about its
axis), so the ceramic is one cylinder (the **outer** wall) with a thinner one
removed (the **cavity**), and the coffee is a third cylinder filling it. The
**handle** is built in the same spirit but explicitly: a small **circle**,
revolved into a **ring** (a torus), stood upright and mounted on the side — then
the slice that would poke into the cup is **trimmed away** (`handle - cavity`).

![The mug as a revolved cross-section; the handle as a revolved circle (ring),
trimmed at the wall.](data/cup-construction.png)

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

def coffee_cup_3d():
    """A ceramic mug (with a ring handle) holding a body of coffee."""
    R, ri, Hout, base, fill = 4.0, 3.4, 9.0, 1.0, 6.0
    outer  = Cylinder(Pnt(0, 0, 0), Z, r=R,  h=Hout, bottom="bottom", mantle="wall")
    cavity = Cylinder(Pnt(0, 0, base), Z, r=ri, h=Hout)        # hollow it out
    ceramic = outer - cavity
    ceramic.faces.Max(Z).name = "rim"

    # a ring handle: revolve a small circle into a torus, stand it up, mount it
    prof = WorkPlane(Axes((0, 0, 0), n=Y)).Circle(1.6, 0, 0.45).Face()
    torus = prof.Revolve(Axis(Pnt(0, 0, 0), Z), 360)          # circle → ring (torus)
    handle = torus.Rotate(Axis(Pnt(0, 0, 0), X), 90).Move((R + 0.9, 0, Hout/2))
    handle = handle - cavity                                  # trim the part poking inside
    handle.faces.name = "handle"
    ceramic = ceramic + handle                                # fuse handle to shell
    ceramic.solids.name = "ceramic"
    ceramic.faces.col = (0.85, 0.83, 0.78)                    # cream ceramic

    coffee = Cylinder(Pnt(0, 0, base), Z, r=ri, h=fill - base, top="surface")
    coffee.solids.name = "coffee"
    coffee.faces.col = (0.40, 0.26, 0.13)                    # brown coffee
    return Glue([ceramic, coffee])

cup = coffee_cup_3d()
Draw(cup)                                                     # the geometry, in colour

# clip with the plane (0,1,0) through the axis, so the interior is visible:
clip3d = {"Clipping": {"enable": True, "function": True, "x": 0, "y": 1, "z": 0, "dist": 0}}
mesh = Mesh(OCCGeometry(cup).GenerateMesh(maxh=1.0)); mesh.Curve(2)
Draw(mesh, settings=clip3d)                                   # ... then the mesh, clipped
print("materials :", mesh.GetMaterials())
print("boundaries:", set(mesh.GetBoundaries()))

## 1. Piecewise coefficients

A `MaterialCF` is a single `CoefficientFunction` that takes a **different value
in each region** — here the heat conductivity $\kappa$ (ceramic conducts more
than the watery coffee). A `BoundaryCF` does the same **on the surface** — here
the heat-transfer coefficient $\alpha$, large where the coffee meets the air,
smaller on the outer wall, and a **small** value everywhere else (rim, handle,
inner wall) — nothing is *perfectly* insulated. Draw them to *see* the
decomposition: the coffee body sits inside the ceramic shell.

In [ ]:
kappa = mesh.MaterialCF({"ceramic": 1.5, "coffee": 0.6})
alpha = mesh.BoundaryCF({"wall": 6.0, "surface": 10.0}, default=1.5)   # small, not zero
# a domain-wise CF lives on volume elements only; to *draw* it (webgui colours the
# surface) sample it into an L2 grid function, which carries a surface trace:
gf_kappa = GridFunction(L2(mesh, order=0)); gf_kappa.Set(kappa)
Draw(gf_kappa, mesh, "κ (material) — clipped to reveal the coffee", settings=clip3d)
Draw(alpha, mesh, "α (heat transfer) — a surface CF", draw_vol=False, draw_surf=True)

## 2. A heat problem with mixed boundary conditions

The mug stands on a **hot coaster** (Dirichlet, $80^\circ$) and loses heat to
the **air** through every other surface by **Newton cooling** ($\alpha\,(T-T_{\!
air})$ — strong at the coffee surface, moderate at the wall, weak at the rim and
handle). The piecewise $\kappa$ and $\alpha$ enter the weak form directly: the
`ds` integral runs over the whole boundary, $\alpha$ supplying the right rate on
each part.

In [ ]:
Tair, Thot = 20.0, 80.0
fes = H1(mesh, order=2, dirichlet="bottom")
u, v = fes.TnT()
a = BilinearForm(kappa*grad(u)*grad(v)*dx + alpha*u*v*ds).Assemble()
f = LinearForm(alpha*Tair*v*ds).Assemble()

gfu = GridFunction(fes)
gfu.Set(Thot, definedon=mesh.Boundaries("bottom"))         # hot coaster
res = f.vec - a.mat * gfu.vec
gfu.vec.data += a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * res
Draw(gfu, mesh, "temperature", settings=clip3d)

## 3. Reading off region quantities

`definedon=mesh.Materials(...)` restricts any integral to one material — so we
can ask for the **average temperature of the coffee** versus the ceramic, or the
coffee's volume, without touching the rest of the mesh.

In [ ]:
def average(region):
    reg = mesh.Materials(region)
    return Integrate(gfu, mesh, definedon=reg) / Integrate(CF(1), mesh, definedon=reg)

print(f"coffee volume       : {Integrate(CF(1), mesh, definedon=mesh.Materials('coffee')):.1f}")
print(f"average T in coffee : {average('coffee'):.1f} °C")
print(f"average T in ceramic: {average('ceramic'):.1f} °C")

The ceramic — a better conductor — runs warmer near the hot coaster, while the
insulating coffee lags behind. Swap the two `kappa` values and watch the picture
flip.

:::{dropdown} 📚 Further reading
:class: further-reading

- i-tutorial [1.5 subdomains](https://docu.ngsolve.org/latest/i-tutorials/unit-1.5-subdomains/subdomains.html).
:::

:::{dropdown} 🧠 Quiz — why is the interface between coffee and ceramic *not* in `mesh.GetBoundaries()`?
:class: quiz
Because `Glue` welds the two solids into one connected body: the shared face
becomes an **internal interface**, not an outer boundary, and the temperature is
automatically continuous across it. Only the *outer* faces — `bottom`, `wall`,
`surface`, `rim` — are boundaries you can put conditions on. If you ever *need*
the interface (say, a contact resistance), you would name it explicitly before
gluing and integrate over it with `dx(skeleton=True)` / an interface region.
:::

Next we put a region's coefficients to work: a steady **heat** problem in the
cup, with Newton cooling and insulation.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "10-steady-heat", "10 · Steady heat in the cup ☕"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))